# NeuraRoads - 03 Performance Analysis

Analyse per-frame performance logs written by the pipeline / benchmark to `src/results/metrics/performance_logs/`. Understand where the frame budget goes and whether the FPS target is met.

In [ ]:
import sys, os
from pathlib import Path
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
SRC = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd() / 'src'
sys.path.insert(0, str(SRC))
from utils.config_loader import PROJECT_ROOT
import pandas as pd
logs = PROJECT_ROOT / 'src' / 'results' / 'metrics' / 'performance_logs'
files = sorted(logs.glob('*.csv'))
print('found logs:', [f.name for f in files])

In [ ]:
# If no log exists yet, generate one with a quick synthetic benchmark.
if not files:
    from pipeline.inference_pipeline import NeuraRoadsPipeline
    import numpy as np
    p = NeuraRoadsPipeline(allow_no_detector=True, frame_rate=30)
    frame = (np.random.rand(720,1280,3)*255).astype('uint8')
    for _ in range(60):
        p.process_frame(frame.copy(), 1/30)
    p.perf.save_csv(logs / 'nb_perf.csv')
    files = sorted(logs.glob('*.csv'))
df = pd.read_csv(files[-1])
df.head()

In [ ]:
import matplotlib.pyplot as plt
print('avg FPS:', round(df['fps'].mean(), 1), '| p95 frame ms:', round(df['total_ms'].quantile(0.95), 1))
plt.figure(figsize=(12, 4))
plt.plot(df['frame'], df['fps'])
plt.axhline(30, color='orange', ls='--', label='30 FPS')
plt.axhline(60, color='green', ls='--', label='60 FPS')
plt.xlabel('frame'); plt.ylabel('FPS'); plt.legend(); plt.title('Instantaneous FPS'); plt.show()

In [ ]:
# Per-stage time breakdown (mean ms/frame).
stage_cols = [c for c in df.columns if c not in ('frame','total_ms','fps')]
means = df[stage_cols].mean().sort_values(ascending=False)
plt.figure(figsize=(9, 4))
means.plot(kind='bar')
plt.ylabel('ms / frame'); plt.title('Per-stage cost'); plt.tight_layout(); plt.show()
means